# Overreliance, Underreliance, and Human-AI Workflow Lab


## Purpose

This lab treats AI use as a workflow, not just a prediction.

Learning goals:

- Simulate accept, override, and escalate decisions.
- Identify where workflow design may encourage overreliance.
- Summarize decision-source behavior.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 1200

workflow = pd.DataFrame({
    "case_id": [f"W-{i:04d}" for i in range(1, n + 1)],
    "model_confidence": rng.beta(5, 2, size=n),
    "explanation_quality": rng.beta(4, 3, size=n),
    "time_pressure": rng.choice(["low", "medium", "high"], size=n, p=[0.35, 0.40, 0.25]),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.40, 0.40, 0.20]),
})

accept_probability = (
    0.10
    + 0.55 * workflow["model_confidence"]
    + 0.15 * workflow["explanation_quality"]
    + workflow["time_pressure"].map({"low": -0.05, "medium": 0.02, "high": 0.12})
    + workflow["risk_level"].map({"low": 0.08, "medium": 0.00, "high": -0.10})
)

workflow["accepted_ai"] = rng.binomial(1, np.clip(accept_probability, 0.02, 0.98), size=n)

workflow["escalated"] = rng.binomial(
    1,
    np.clip(
        0.05
        + 0.25 * (workflow["risk_level"] == "high").astype(int)
        + 0.20 * (workflow["explanation_quality"] < 0.45).astype(int),
        0.01,
        0.90,
    ),
    size=n,
)

workflow["workflow_outcome"] = np.select(
    [
        workflow["escalated"] == 1,
        workflow["accepted_ai"] == 1,
    ],
    [
        "escalated",
        "accepted_ai",
    ],
    default="overrode_or_ignored",
)

workflow.groupby(["time_pressure", "risk_level", "workflow_outcome"]).size().reset_index(name="cases")

## Interpretation

Workflow pressure affects reliance. A system can produce the same output but lead to different human decisions under different operational conditions.